In [358]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn import preprocessing

# Importação de Dados

Neste ponto importamos os dados obtidos com o script `coleta_dados.py` e observamos algumas informações básicas do dataset, pode-se observar que há um número elevado de linhas com dados nulos.

In [359]:
df = pd.read_csv("../data/dados_paises_com_regiao.csv")
df.sort_values(by=['Country_Code', 'Year'], inplace=True)
df.info()

<class 'pandas.DataFrame'>
Index: 3255 entries, 1455 to 3254
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Country_Name             3255 non-null   str    
 1   Country_Code             3255 non-null   str    
 2   Region                   3255 non-null   str    
 3   Income_Group             3255 non-null   str    
 4   Year                     3255 non-null   int64  
 5   Internet_Usage_Pct       2894 non-null   float64
 6   GDP_Per_Capita_PPP       2976 non-null   float64
 7   Broadband_Subscriptions  2900 non-null   float64
 8   GDP_USD                  3144 non-null   float64
 9   Secure_Servers           3193 non-null   float64
dtypes: float64(5), int64(1), str(4)
memory usage: 444.1 KB


# Tratamento dos Dados

## Interpolação de Valores Nulos

Para resolver o problema dos dados nulos será usado a técnica de interpolação linear aplicada trés vezes com agrupamentos diferentes, primeiro agrupamos observações do próprio país, depois agrupamos as observações de um ano de países da mesma região e grupo de renda, e por último agrupamos apenas por ano e região. As interpolações foram aplicadas nos agrupamentos na ordem que eles foram mencionados anteriormente para garantir o maior nível de precisão.

In [360]:
indicadores = {
    'Internet_Usage_Pct': 'Uso de Internet',
    'GDP_Per_Capita_PPP': 'PIB per Capita',
    'Broadband_Subscriptions': 'Banda Larga Fixa',
    'GDP_USD': 'Tamanho Economia PIB',
    'Secure_Servers': 'Infra Servidores B2B'
}

df_interpolado = df

for col in indicadores.keys():
    # O limit_direction='both' garante que ele preencha tanto buracos no meio, 
    # quanto buracos nas pontas (usando o primeiro ou último valor disponível)
    df_interpolado[col] = df_interpolado.groupby('Country_Code')[col].transform(
        lambda x:  x.interpolate(method='linear', limit_direction='both')
    )

for col in indicadores.keys():
    # O limit_direction='both' garante que ele preencha tanto buracos no meio, 
    # quanto buracos nas pontas (usando o primeiro ou último valor disponível)
    df_interpolado[col] = df_interpolado.groupby(['Region', 'Income_Group', 'Year'])[col].transform(
        lambda x:  x.interpolate(method='linear', limit_direction='both')
    )

for col in indicadores.keys():
    # O limit_direction='both' garante que ele preencha tanto buracos no meio, 
    # quanto buracos nas pontas (usando o primeiro ou último valor disponível)
    df_interpolado[col] = df_interpolado.groupby(['Region', 'Year'])[col].transform(
        lambda x:  x.interpolate(method='linear', limit_direction='both')
    )

df_interpolado.info()

<class 'pandas.DataFrame'>
Index: 3255 entries, 1455 to 3254
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Country_Name             3255 non-null   str    
 1   Country_Code             3255 non-null   str    
 2   Region                   3255 non-null   str    
 3   Income_Group             3255 non-null   str    
 4   Year                     3255 non-null   int64  
 5   Internet_Usage_Pct       3255 non-null   float64
 6   GDP_Per_Capita_PPP       3255 non-null   float64
 7   Broadband_Subscriptions  3255 non-null   float64
 8   GDP_USD                  3255 non-null   float64
 9   Secure_Servers           3255 non-null   float64
dtypes: float64(5), int64(1), str(4)
memory usage: 444.1 KB


## Filtragem de Países da OCDE

Para juntar os dados do Banco Mundial junto com os dados da OCDE iremos retirar da tabela os dados referentes a países que não são representados nos dados da OCDE.

In [361]:
ocde = pd.read_csv("../data/dados_ocde_software.csv")
ocde = ocde["Pais"].unique()

df_filtrado = df_interpolado.loc[df_interpolado["Country_Code"].isin(ocde)]

df_filtrado.reset_index(drop=True, inplace=True)
df_filtrado.info()

<class 'pandas.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Country_Name             645 non-null    str    
 1   Country_Code             645 non-null    str    
 2   Region                   645 non-null    str    
 3   Income_Group             645 non-null    str    
 4   Year                     645 non-null    int64  
 5   Internet_Usage_Pct       645 non-null    float64
 6   GDP_Per_Capita_PPP       645 non-null    float64
 7   Broadband_Subscriptions  645 non-null    float64
 8   GDP_USD                  645 non-null    float64
 9   Secure_Servers           645 non-null    float64
dtypes: float64(5), int64(1), str(4)
memory usage: 79.8 KB


## Normalização de Dados Numéricos

Como pode ser visto abaixo os diversos indicadores estão em escalas bem diferentes um dos outros, com um indicador com mínimo e máximo percentual e os outros com números que podem chegar até 10^13. Além disso, como descoberto na exploração de dados, existem diversos outliers no banco de dados, porém interpretamos que não seria apropriado simplesmente retirar esses outliers pois representam diferenças estruturais entre países ao invés de serem dados discrepantes.

Visto isso, num primeiro momento, utilizaremos a Normalização Robust por causa da sua resistência a outliers e sua ampla utilização em domínios relacionados a economia.

In [362]:
df_filtrado.describe()

,Year,Internet_Usage_Pct,GDP_Per_Capita_PPP,Broadband_Subscriptions,GDP_USD,Secure_Servers
count,645.000000,645.000000,645.000000,645.000000,6.450000e+02,645.000000
mean,2017.000000,79.747769,42438.081016,29.934155,1.073717e+12,25479.102813
std,4.323847,14.308059,22938.925528,9.617983,3.174752e+12,48633.281387
min,2010.000000,36.500000,9085.742942,3.603585,9.097044e+09,4.119435
25%,2013.000000,71.400000,26607.725680,23.708371,5.512207e+10,947.719181
50%,2017.000000,83.241176,39839.977277,30.173651,2.560559e+11,4823.183120
75%,2021.000000,90.763555,52517.335491,37.434553,7.062349e+11,29131.217625
max,2024.000000,99.769152,155941.288211,48.925198,2.875096e+13,557174.254410


In [ ]:
scaler = preprocessing.RobustScaler()

x_normal = scaler.fit_transform(df_filtrado[indicadores.keys()])
x_normal = pd.DataFrame(x_normal, columns=indicadores.keys(), index=df_filtrado.index)


df_normalizado = df_filtrado[["Country_Name", "Country_Code", "Region", "Income_Group", "Year"]]
df_normalizado = pd.concat([df_normalizado, x_normal], axis=1)
df_normalizado.info()
df_normalizado.describe()

<class 'pandas.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Internet_Usage_Pct       645 non-null    float64
 1   GDP_Per_Capita_PPP       645 non-null    float64
 2   Broadband_Subscriptions  645 non-null    float64
 3   GDP_USD                  645 non-null    float64
 4   Secure_Servers           645 non-null    float64
dtypes: float64(5)
memory usage: 25.3 KB
<class 'pandas.DataFrame'>
RangeIndex: 645 entries, 0 to 644
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Country_Name             645 non-null    str    
 1   Country_Code             645 non-null    str    
 2   Region                   645 non-null    str    
 3   Income_Group             645 non-null    str    
 4   Year                     645 non-null    int64  
 5   Internet_Us

,Year,Internet_Usage_Pct,GDP_Per_Capita_PPP,Broadband_Subscriptions,GDP_USD,Secure_Servers
count,645.000000,645.000000,645.000000,645.000000,645.000000,645.000000
mean,2017.000000,-0.180411,0.100276,-0.017448,1.255790,0.732908
std,4.323847,0.738917,0.885344,0.700703,4.875886,1.725594
min,2010.000000,-2.413874,-1.186982,-1.935722,-0.379287,-0.170989
25%,2013.000000,-0.611519,-0.510708,-0.471018,-0.308601,-0.137508
50%,2017.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,2021.000000,0.388481,0.489292,0.528982,0.691399,0.862492
max,2024.000000,0.853561,4.481013,1.366115,43.763380,19.598386
